In [1]:
# Import libraries and define configurations
import json
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
import joblib
from sklearn import set_config
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

RANDOM_STATE = 121
N_JOBS = -1

np.random.seed(RANDOM_STATE)


CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_WORKING_DIRECTORY.parent
    if CURRENT_WORKING_DIRECTORY.name == "notebooks"
    else CURRENT_WORKING_DIRECTORY
)

PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"
REPORTS_DIRECTORY = PROJECT_ROOT / "reports"
MODELS_DIRECTORY = PROJECT_ROOT / "models"

ENGINEERED_FEATURES_PARQUET_PATH = (
    PROCESSED_DATA_DIRECTORY / "donor_features.parquet"
)
ENGINEERED_FEATURES_CSV_PATH = (
    PROCESSED_DATA_DIRECTORY / "donor_features.csv"
)
CLEANED_DONOR_DATA_PATH = (
    PROCESSED_DATA_DIRECTORY / "cleaned_donor_data.csv"
)
FEATURE_DICTIONARY_PATH = (
    REPORTS_DIRECTORY / "feature_dictionary.csv"
)

MODEL_PREDICTIONS_PATH = (
    PROCESSED_DATA_DIRECTORY / "model_predictions.csv"
)
FINAL_PRIMARY_PIPELINE_PATH = (
    MODELS_DIRECTORY / "final_primary_donor_pipeline.joblib"
)
MODEL_COMPARISON_RESULTS_PATH = (
    REPORTS_DIRECTORY / "05_model_comparison_results.csv"
)
CLASSIFICATION_MODELING_REPORT_PATH = (
    REPORTS_DIRECTORY / "05_classification_modeling.md"
)

MODELS_DIRECTORY.mkdir(parents=True, exist_ok=True)
REPORTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(style="whitegrid")
set_config(display="diagram")

warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# Load the engineered dataset and feature dictionary
def format_project_path(path):
    relative_path = Path(path).resolve().relative_to(PROJECT_ROOT)
    return f"/{PROJECT_ROOT.name}/{relative_path.as_posix()}"


artifact_availability = pd.DataFrame({
    "artifact": [
        "Engineered features Parquet",
        "Engineered features CSV fallback",
        "Feature dictionary",
        "Cleaned donor dataset",
    ],
    "path_object": [
        ENGINEERED_FEATURES_PARQUET_PATH,
        ENGINEERED_FEATURES_CSV_PATH,
        FEATURE_DICTIONARY_PATH,
        CLEANED_DONOR_DATA_PATH,
    ],
})

artifact_availability["path"] = artifact_availability["path_object"].apply(
    format_project_path
)
artifact_availability["available"] = artifact_availability["path_object"].apply(
    Path.exists
)

display(artifact_availability[
    ["artifact", "path", "available"]
].style
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
)

if ENGINEERED_FEATURES_PARQUET_PATH.exists():
    modeling_data = pd.read_parquet(
        ENGINEERED_FEATURES_PARQUET_PATH
    )
    modeling_data_source = "Parquet"
    modeling_data_path = ENGINEERED_FEATURES_PARQUET_PATH

elif ENGINEERED_FEATURES_CSV_PATH.exists():
    modeling_data = pd.read_csv(
        ENGINEERED_FEATURES_CSV_PATH
    )
    modeling_data_source = "CSV fallback"
    modeling_data_path = ENGINEERED_FEATURES_CSV_PATH

else:
    raise FileNotFoundError(
        "Neither donor_features.parquet nor donor_features.csv "
        "was found in the processed data directory."
    )

if not FEATURE_DICTIONARY_PATH.exists():
    raise FileNotFoundError(
        f"Feature dictionary not found: "
        f"{format_project_path(FEATURE_DICTIONARY_PATH)}"
    )

feature_dictionary = pd.read_csv(
    FEATURE_DICTIONARY_PATH
)

print(f"\nModeling dataset source: {modeling_data_source}")
print(
    "Modeling dataset path:",
    format_project_path(modeling_data_path),
)
print(
    "Feature dictionary path:",
    format_project_path(FEATURE_DICTIONARY_PATH),
)

print(
    "\nModeling dataset loaded:",
    f"{modeling_data.shape[0]:,} rows × "
    f"{modeling_data.shape[1]:,} columns",
)

print(
    "Feature dictionary loaded:",
    f"{feature_dictionary.shape[0]:,} rows × "
    f"{feature_dictionary.shape[1]:,} columns\n",
)

artifact,path,available
Engineered features Parquet,/red-cross-donor-prediction/data/processed/donor_features.parquet,True
Engineered features CSV fallback,/red-cross-donor-prediction/data/processed/donor_features.csv,True
Feature dictionary,/red-cross-donor-prediction/reports/feature_dictionary.csv,True
Cleaned donor dataset,/red-cross-donor-prediction/data/processed/cleaned_donor_data.csv,True



Modeling dataset source: Parquet
Modeling dataset path: /red-cross-donor-prediction/data/processed/donor_features.parquet
Feature dictionary path: /red-cross-donor-prediction/reports/feature_dictionary.csv

Modeling dataset loaded: 34,403 rows × 55 columns
Feature dictionary loaded: 77 rows × 7 columns



In [3]:
# Validate the Phase 4 modeling export
EXPECTED_ROW_COUNT = 34_403
EXPECTED_COLUMN_COUNT = 55
EXPECTED_PREDICTOR_COUNT = 53

TRACKING_IDENTIFIER_COLUMN = "donor_unique_id"
PRIMARY_TARGET_COLUMN = "target_current_fiscal_year_donor_flag"

DIRECT_LEAKAGE_COLUMNS = {
    "current_fiscal_year_donation",
    "cumulative_donation_amount",
}

EXPECTED_TARGET_COUNTS = {
    0: 32_499,
    1: 1_904,
}

excluded_modeling_columns = {
    TRACKING_IDENTIFIER_COLUMN,
    PRIMARY_TARGET_COLUMN,
}

predictor_columns = [
    column
    for column in modeling_data.columns
    if column not in excluded_modeling_columns
]

unexpected_direct_leakage_columns = sorted(
    set(predictor_columns).intersection(DIRECT_LEAKAGE_COLUMNS)
)

numeric_columns = modeling_data.select_dtypes(
    include=[np.number]
).columns

infinite_value_count = int(
    np.isinf(modeling_data[numeric_columns]).sum().sum()
)

target_distribution = (
    modeling_data[PRIMARY_TARGET_COLUMN]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis("target_class")
    .reset_index(name="record_count")
)

target_distribution["percentage"] = (
    target_distribution["record_count"]
    / len(modeling_data)
    * 100
)

target_distribution["expected_record_count"] = (
    target_distribution["target_class"]
    .map(EXPECTED_TARGET_COUNTS)
)

target_distribution["matches_expected"] = (
    target_distribution["record_count"]
    == target_distribution["expected_record_count"]
)

actual_target_counts = target_distribution.set_index(
    "target_class"
)["record_count"].to_dict()

validation_results = pd.DataFrame({
    "validation_check": [
        "Record count",
        "Total column count",
        "Predictor count",
        "Tracking identifier count",
        "Primary target count",
        "Missing tracking identifiers",
        "Duplicate tracking identifiers",
        "Unexpected direct-leakage columns",
        "Infinite numeric values",
        "Primary target class 0 count",
        "Primary target class 1 count",
    ],
    "expected": [
        EXPECTED_ROW_COUNT,
        EXPECTED_COLUMN_COUNT,
        EXPECTED_PREDICTOR_COUNT,
        1,
        1,
        0,
        0,
        "None",
        0,
        EXPECTED_TARGET_COUNTS[0],
        EXPECTED_TARGET_COUNTS[1],
    ],
    "actual": [
        modeling_data.shape[0],
        modeling_data.shape[1],
        len(predictor_columns),
        list(modeling_data.columns).count(
            TRACKING_IDENTIFIER_COLUMN
        ),
        list(modeling_data.columns).count(
            PRIMARY_TARGET_COLUMN
        ),
        modeling_data[
            TRACKING_IDENTIFIER_COLUMN
        ].isna().sum(),
        modeling_data[
            TRACKING_IDENTIFIER_COLUMN
        ].duplicated().sum(),
        (
            ", ".join(unexpected_direct_leakage_columns)
            if unexpected_direct_leakage_columns
            else "None"
        ),
        infinite_value_count,
        actual_target_counts[0],
        actual_target_counts[1],
    ],
})

validation_results["passed"] = [
    modeling_data.shape[0] == EXPECTED_ROW_COUNT,
    modeling_data.shape[1] == EXPECTED_COLUMN_COUNT,
    len(predictor_columns) == EXPECTED_PREDICTOR_COUNT,
    list(modeling_data.columns).count(
        TRACKING_IDENTIFIER_COLUMN
    ) == 1,
    list(modeling_data.columns).count(
        PRIMARY_TARGET_COLUMN
    ) == 1,
    modeling_data[
        TRACKING_IDENTIFIER_COLUMN
    ].isna().sum() == 0,
    modeling_data[
        TRACKING_IDENTIFIER_COLUMN
    ].duplicated().sum() == 0,
    len(unexpected_direct_leakage_columns) == 0,
    infinite_value_count == 0,
    actual_target_counts[0] == EXPECTED_TARGET_COUNTS[0],
    actual_target_counts[1] == EXPECTED_TARGET_COUNTS[1],
]

def format_validation_value(value):
    if isinstance(value, (bool, np.bool_)):
        return str(value)

    if isinstance(value, (int, np.integer)):
        return f"{value:,.0f}"

    if isinstance(value, (float, np.floating)):
        if pd.isna(value):
            return ""

        if value.is_integer():
            return f"{value:,.0f}"

        if value != 0 and abs(value) < 0.01:
            return f"{value:,.6f}".rstrip("0").rstrip(".")

        return f"{value:,.2f}"

    return str(value)


display(validation_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

print("")

display(target_distribution.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
        "expected_record_count": "{:,.0f}",
    })
)

failed_validation_checks = validation_results.loc[
    ~validation_results["passed"],
    "validation_check",
].tolist()

if failed_validation_checks:
    raise AssertionError(
        "Phase 4 modeling export validation failed for: "
        + ", ".join(failed_validation_checks)
    )

print("\nAll Phase 4 modeling export validation checks passed.\n")

validation_check,expected,actual,passed
Record count,"34,403","34,403",True
Total column count,55,55,True
Predictor count,53,53,True
Tracking identifier count,1,1,True
Primary target count,1,1,True
Missing tracking identifiers,0,0,True
Duplicate tracking identifiers,0,0,True
Unexpected direct-leakage columns,None,None,True
Infinite numeric values,0,0,True
Primary target class 0 count,"32,499","32,499",True


target_class,record_count,percentage,expected_record_count,matches_expected
0,"32,499",94.47%,"32,499",True
1,"1,904",5.53%,"1,904",True



All Phase 4 modeling export validation checks passed.



## Modeling Setup and Data Validation

The Phase 5 modeling environment was configured using a consistent random state of `121`, reusable project paths, and the libraries required for preprocessing, classification, model evaluation, visualization, and model persistence.

Project paths are defined relative to the repository root so the notebook does not display user-specific local directories.

The engineered modeling dataset was loaded from the Parquet export, with the CSV retained as a fallback. The feature dictionary and original cleaned dataset were also confirmed to be available.

The modeling export contains:

* 34,403 records
* 55 total columns
* 53 leakage-safe predictors
* 1 tracking identifier
* 1 primary target

The `donor_unique_id` field contains no missing or duplicate values. No direct-leakage columns or infinite numeric values were found.

The primary target distribution remains unchanged:

| Target Class | Records | Percentage |
| ------------ | ------: | ---------: |
| 0            |  32,499 |     94.47% |
| 1            |   1,904 |      5.53% |

All Phase 4 modeling export validation checks passed. The severe class imbalance confirms that Phase 5 should use stratified splitting and evaluation metrics beyond accuracy.


In [4]:
# Define the primary target, tracking identifier, and predictor matrix
tracking_donor_ids = modeling_data[
    TRACKING_IDENTIFIER_COLUMN
].copy()

target_primary_donor_flag = modeling_data[
    PRIMARY_TARGET_COLUMN
].copy()

features_primary_model = modeling_data.drop(
    columns=[
        TRACKING_IDENTIFIER_COLUMN,
        PRIMARY_TARGET_COLUMN,
    ]
).copy()

primary_model_objects_summary = pd.DataFrame({
    "object_name": [
        "tracking_donor_ids",
        "target_primary_donor_flag",
        "features_primary_model",
    ],
    "record_count": [
        tracking_donor_ids.shape[0],
        target_primary_donor_flag.shape[0],
        features_primary_model.shape[0],
    ],
    "column_count": [
        1,
        1,
        features_primary_model.shape[1],
    ],
})

display(primary_model_objects_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["object_name"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "record_count": "{:,.0f}",
        "column_count": "{:,.0f}",
    })
)

assert len(tracking_donor_ids) == len(modeling_data)
assert len(target_primary_donor_flag) == len(modeling_data)
assert len(features_primary_model) == len(modeling_data)

assert tracking_donor_ids.index.equals(
    target_primary_donor_flag.index
)
assert tracking_donor_ids.index.equals(
    features_primary_model.index
)

assert TRACKING_IDENTIFIER_COLUMN not in features_primary_model.columns
assert PRIMARY_TARGET_COLUMN not in features_primary_model.columns
assert features_primary_model.shape[1] == EXPECTED_PREDICTOR_COUNT

print(
    "\nPrimary modeling objects were created and validated successfully."
)

object_name,record_count,column_count
tracking_donor_ids,"34,403",1
target_primary_donor_flag,"34,403",1
features_primary_model,"34,403",53



Primary modeling objects were created and validated successfully.


In [5]:
# Build and validate predictor list
required_feature_dictionary_columns = {
    "feature_name",
    "leakage_status",
}

missing_feature_dictionary_columns = sorted(
    required_feature_dictionary_columns
    - set(feature_dictionary.columns)
)

if missing_feature_dictionary_columns:
    raise KeyError(
        "Missing required feature dictionary columns: "
        + ", ".join(missing_feature_dictionary_columns)
    )

feature_dictionary_leakage_status = (
    feature_dictionary["leakage_status"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

leakage_safe_predictor_columns = feature_dictionary.loc[
    feature_dictionary_leakage_status.eq("safe"),
    "feature_name",
].tolist()

non_safe_dictionary_features = set(
    feature_dictionary.loc[
        ~feature_dictionary_leakage_status.eq("safe"),
        "feature_name",
    ]
)

missing_safe_predictors = sorted(
    set(leakage_safe_predictor_columns)
    - set(features_primary_model.columns)
)

unexpected_predictor_columns = sorted(
    set(features_primary_model.columns)
    - set(leakage_safe_predictor_columns)
)

duplicate_safe_predictors = sorted(
    pd.Series(leakage_safe_predictor_columns)[
        pd.Series(leakage_safe_predictor_columns).duplicated()
    ].unique()
)

duplicate_predictor_matrix_columns = sorted(
    features_primary_model.columns[
        features_primary_model.columns.duplicated()
    ].unique()
)

identifier_or_target_predictors = sorted(
    set(leakage_safe_predictor_columns).intersection({
        TRACKING_IDENTIFIER_COLUMN,
        PRIMARY_TARGET_COLUMN,
    })
)

non_safe_predictors_included = sorted(
    set(leakage_safe_predictor_columns).intersection(
        non_safe_dictionary_features
    )
)

predictor_validation_results = pd.DataFrame({
    "validation_check": [
        "Safe predictors exist in modeling export",
        "No unexpected predictors in modeling matrix",
        "No duplicate names in safe predictor list",
        "No duplicate columns in predictor matrix",
        "No identifier or target included",
        "No excluded or timing-sensitive features included",
        "Final safe predictor count",
    ],
    "expected": [
        "None missing",
        "None",
        0,
        0,
        "None",
        "None",
        EXPECTED_PREDICTOR_COUNT,
    ],
    "actual": [
        (
            ", ".join(missing_safe_predictors)
            if missing_safe_predictors
            else "None missing"
        ),
        (
            ", ".join(unexpected_predictor_columns)
            if unexpected_predictor_columns
            else "None"
        ),
        len(duplicate_safe_predictors),
        len(duplicate_predictor_matrix_columns),
        (
            ", ".join(identifier_or_target_predictors)
            if identifier_or_target_predictors
            else "None"
        ),
        (
            ", ".join(non_safe_predictors_included)
            if non_safe_predictors_included
            else "None"
        ),
        len(leakage_safe_predictor_columns),
    ],
})

predictor_validation_results["passed"] = [
    len(missing_safe_predictors) == 0,
    len(unexpected_predictor_columns) == 0,
    len(duplicate_safe_predictors) == 0,
    len(duplicate_predictor_matrix_columns) == 0,
    len(identifier_or_target_predictors) == 0,
    len(non_safe_predictors_included) == 0,
    len(leakage_safe_predictor_columns) == EXPECTED_PREDICTOR_COUNT,
]

display(predictor_validation_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

failed_predictor_checks = predictor_validation_results.loc[
    ~predictor_validation_results["passed"],
    "validation_check",
].tolist()

if failed_predictor_checks:
    raise AssertionError(
        "Leakage-safe predictor validation failed for: "
        + ", ".join(failed_predictor_checks)
    )

features_primary_model = features_primary_model.loc[
    :,
    leakage_safe_predictor_columns,
].copy()

print(
    f"\nLeakage-safe predictor list validated with "
    f"{len(leakage_safe_predictor_columns):,} features.\n"
)

validation_check,expected,actual,passed
Safe predictors exist in modeling export,None missing,None missing,True
No unexpected predictors in modeling matrix,None,None,True
No duplicate names in safe predictor list,0,0,True
No duplicate columns in predictor matrix,0,0,True
No identifier or target included,None,None,True
No excluded or timing-sensitive features included,None,None,True
Final safe predictor count,53,53,True



Leakage-safe predictor list validated with 53 features.



In [6]:
# Recreate and validate feature set variants
safe_baseline_historical_features = [
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
    "is_alumnus_flag",
    "is_parent_flag",
    "donor_age",
    "feature_gender_identity",
    "feature_preferred_address_type",
]

safe_aggregate_rfm_features = [
    "feature_past_5yr_total_donation",
    "feature_past_5yr_average_donation",
    "feature_past_5yr_donation_frequency_rate",
    "feature_years_since_last_donation_past_5yr",
    "feature_past_5yr_max_donation",
    "donor_age",
    "feature_gender_identity",
    "feature_preferred_address_type",
]

safe_trend_enhanced_features = [
    "feature_past_5yr_total_donation",
    "feature_past_5yr_average_donation",
    "feature_past_5yr_donation_frequency_rate",
    "feature_years_since_last_donation_past_5yr",
    "feature_past_5yr_max_donation",
    "donor_age",
    "feature_gender_identity",
    "feature_preferred_address_type",
    "feature_recent_vs_older_donation_difference",
    "feature_recent_vs_older_donation_ratio",
    "feature_past_5yr_donation_trend_slope",
    "feature_last_year_donation_amount_change",
    "feature_past_5yr_donation_coefficient_of_variation",
    "feature_last_vs_previous_donation_ratio",
    "feature_log1p_last_vs_previous_donation_ratio",
]

full_leakage_safe_candidate_features = [
    "donor_age",
    "is_alumnus_flag",
    "is_parent_flag",
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
    "feature_past_5yr_total_donation",
    "feature_past_5yr_average_donation",
    "feature_past_5yr_median_donation",
    "feature_past_5yr_max_donation",
    "feature_past_5yr_min_donation",
    "feature_past_5yr_donation_std",
    "feature_past_5yr_active_average_donation",
    "feature_years_donated_past_5yr",
    "feature_past_5yr_donation_frequency_rate",
    "feature_any_past_donation_flag",
    "feature_multiple_year_donor_flag",
    "feature_consistent_donor_flag",
    "feature_max_donation_streak_past_5yr",
    "feature_intermittent_donor_flag",
    "feature_years_since_last_donation_past_5yr",
    "feature_donated_last_year_flag",
    "feature_donated_within_2_years_flag",
    "feature_lapsed_donor_flag",
    "feature_never_donated_past_5yr_flag",
    "feature_most_recent_positive_donation",
    "feature_recent_vs_older_donation_difference",
    "feature_recent_vs_older_donation_ratio",
    "feature_past_5yr_donation_trend_slope",
    "feature_last_year_donation_amount_change",
    "feature_past_5yr_donation_coefficient_of_variation",
    "feature_previous_year_donation_zero_flag",
    "feature_last_vs_previous_donation_ratio",
    "feature_log1p_last_vs_previous_donation_ratio",
    "feature_log_past_5yr_total_donation",
    "feature_log_past_5yr_average_donation",
    "feature_log_past_5yr_max_donation",
    "feature_log_most_recent_positive_donation",
    "feature_age_group",
    "feature_age_decade",
    "feature_age_squared",
    "feature_gender_identity",
    "feature_gender_missing_flag",
    "feature_gender_unknown_flag",
    "feature_preferred_address_type",
    "feature_preferred_address_missing_flag",
    "feature_is_business_address",
    "feature_is_home_address",
    "feature_is_campus_address",
    "feature_address_type_other",
    "feature_postal_code_missing_flag",
]

feature_set_variants = {
    "Safe Baseline Historical Set": safe_baseline_historical_features,
    "Safe Aggregate RFM Set": safe_aggregate_rfm_features,
    "Safe Trend-Enhanced Set": safe_trend_enhanced_features,
    "Full Leakage-Safe Candidate Set": full_leakage_safe_candidate_features,
}

expected_feature_set_counts = {
    "Safe Baseline Historical Set": 10,
    "Safe Aggregate RFM Set": 8,
    "Safe Trend-Enhanced Set": 15,
    "Full Leakage-Safe Candidate Set": 53,
}

safe_dictionary_features = set(
    feature_dictionary.loc[
        feature_dictionary["leakage_status"]
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq("safe"),
        "feature_name",
    ]
)

feature_set_validation_records = []
feature_set_validation_details = {}

for feature_set_name, feature_list in feature_set_variants.items():
    duplicate_features = sorted(
        pd.Series(feature_list)[
            pd.Series(feature_list).duplicated()
        ].unique()
    )

    missing_from_model = sorted(
        set(feature_list) - set(features_primary_model.columns)
    )

    missing_from_dictionary = sorted(
        set(feature_list) - set(feature_dictionary["feature_name"])
    )

    non_safe_features = sorted(
        set(feature_list) - safe_dictionary_features
    )

    forbidden_features = sorted(
        set(feature_list).intersection({
            TRACKING_IDENTIFIER_COLUMN,
            PRIMARY_TARGET_COLUMN,
        })
    )

    outside_full_safe_list = sorted(
        set(feature_list) - set(leakage_safe_predictor_columns)
    )

    expected_count = expected_feature_set_counts[feature_set_name]

    feature_set_passed = all([
        len(feature_list) == expected_count,
        len(duplicate_features) == 0,
        len(missing_from_model) == 0,
        len(missing_from_dictionary) == 0,
        len(non_safe_features) == 0,
        len(forbidden_features) == 0,
        len(outside_full_safe_list) == 0,
    ])

    feature_set_validation_records.append({
        "feature_set": feature_set_name,
        "expected_count": expected_count,
        "actual_count": len(feature_list),
        "duplicate_count": len(duplicate_features),
        "missing_model_count": len(missing_from_model),
        "missing_dictionary_count": len(missing_from_dictionary),
        "non_safe_count": len(non_safe_features),
        "forbidden_count": len(forbidden_features),
        "passed": feature_set_passed,
    })

    feature_set_validation_details[feature_set_name] = {
        "duplicates": duplicate_features,
        "missing_from_model": missing_from_model,
        "missing_from_dictionary": missing_from_dictionary,
        "non_safe_features": non_safe_features,
        "forbidden_features": forbidden_features,
        "outside_full_safe_list": outside_full_safe_list,
    }

feature_set_validation_results = pd.DataFrame(
    feature_set_validation_records
)

full_candidate_matches_safe_export = (
    set(full_leakage_safe_candidate_features)
    == set(leakage_safe_predictor_columns)
)

display(feature_set_validation_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["feature_set"],
        **{"text-align": "center"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected_count": "{:,.0f}",
        "actual_count": "{:,.0f}",
        "duplicate_count": "{:,.0f}",
        "missing_model_count": "{:,.0f}",
        "missing_dictionary_count": "{:,.0f}",
        "non_safe_count": "{:,.0f}",
        "forbidden_count": "{:,.0f}",
    })
)

failed_feature_sets = feature_set_validation_results.loc[
    ~feature_set_validation_results["passed"],
    "feature_set",
].tolist()

if failed_feature_sets:
    failed_details = {
        feature_set_name: feature_set_validation_details[feature_set_name]
        for feature_set_name in failed_feature_sets
    }

    raise AssertionError(
        "Feature-set validation failed: "
        + json.dumps(failed_details, indent=2)
    )

if not full_candidate_matches_safe_export:
    raise AssertionError(
        "The Full Leakage-Safe Candidate Set does not match "
        "the 53-feature safe predictor list."
    )

print(
    "\nAll leakage-safe feature-set variants were recreated "
    "and validated successfully."
)

feature_set,expected_count,actual_count,duplicate_count,missing_model_count,missing_dictionary_count,non_safe_count,forbidden_count,passed
Safe Baseline Historical Set,10,10,0,0,0,0,0,True
Safe Aggregate RFM Set,8,8,0,0,0,0,0,True
Safe Trend-Enhanced Set,15,15,0,0,0,0,0,True
Full Leakage-Safe Candidate Set,53,53,0,0,0,0,0,True



All leakage-safe feature-set variants were recreated and validated successfully.


In [7]:
# Classify predictors by primary data type and modeling role
numeric_continuous_features = [
    "donor_age",
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
    "feature_past_5yr_total_donation",
    "feature_past_5yr_average_donation",
    "feature_past_5yr_median_donation",
    "feature_past_5yr_max_donation",
    "feature_past_5yr_min_donation",
    "feature_past_5yr_donation_std",
    "feature_past_5yr_active_average_donation",
    "feature_most_recent_positive_donation",
    "feature_recent_vs_older_donation_difference",
    "feature_recent_vs_older_donation_ratio",
    "feature_past_5yr_donation_trend_slope",
    "feature_last_year_donation_amount_change",
    "feature_past_5yr_donation_coefficient_of_variation",
    "feature_last_vs_previous_donation_ratio",
    "feature_log1p_last_vs_previous_donation_ratio",
    "feature_log_past_5yr_total_donation",
    "feature_log_past_5yr_average_donation",
    "feature_log_past_5yr_max_donation",
    "feature_log_most_recent_positive_donation",
    "feature_age_squared",
]

numeric_binary_features = [
    "is_alumnus_flag",
    "is_parent_flag",
    "feature_any_past_donation_flag",
    "feature_multiple_year_donor_flag",
    "feature_consistent_donor_flag",
    "feature_intermittent_donor_flag",
    "feature_donated_last_year_flag",
    "feature_donated_within_2_years_flag",
    "feature_lapsed_donor_flag",
    "feature_never_donated_past_5yr_flag",
    "feature_previous_year_donation_zero_flag",
    "feature_gender_missing_flag",
    "feature_gender_unknown_flag",
    "feature_preferred_address_missing_flag",
    "feature_is_business_address",
    "feature_is_home_address",
    "feature_is_campus_address",
    "feature_address_type_other",
    "feature_postal_code_missing_flag",
]

ordinal_discrete_numeric_features = [
    "feature_years_donated_past_5yr",
    "feature_past_5yr_donation_frequency_rate",
    "feature_max_donation_streak_past_5yr",
    "feature_years_since_last_donation_past_5yr",
]

categorical_features = [
    "feature_age_group",
    "feature_age_decade",
    "feature_gender_identity",
    "feature_preferred_address_type",
]

structurally_missing_features = [
    "feature_recent_vs_older_donation_ratio",
]

near_constant_features = [
    "feature_past_5yr_min_donation",
    "feature_address_type_other",
    "feature_postal_code_missing_flag",
]

primary_feature_type_groups = {
    "Numeric continuous": numeric_continuous_features,
    "Numeric binary": numeric_binary_features,
    "Ordinal or discrete numeric": ordinal_discrete_numeric_features,
    "Categorical": categorical_features,
}

feature_to_primary_type = {
    feature_name: feature_type
    for feature_type, feature_list in primary_feature_type_groups.items()
    for feature_name in feature_list
}

special_consideration_map = {
    "feature_recent_vs_older_donation_ratio": (
        "Structural missingness; impute within the pipeline and "
        "consider a missingness indicator"
    ),
    "feature_years_since_last_donation_past_5yr": (
        "Value 6 is a sentinel for no donation in the five-year window"
    ),
    "feature_past_5yr_min_donation": (
        "Near constant; compare retention and removal by model"
    ),
    "feature_address_type_other": (
        "Near constant; compare retention and removal by model"
    ),
    "feature_postal_code_missing_flag": (
        "Near constant; compare retention and removal by model"
    ),
    "feature_last_vs_previous_donation_ratio": (
        "May contain extreme values when the previous donation is small"
    ),
    "feature_log1p_last_vs_previous_donation_ratio": (
        "Log-transformed version retained for model-specific comparison"
    ),
}

predictor_modeling_roles = pd.DataFrame({
    "feature_name": leakage_safe_predictor_columns,
})

predictor_modeling_roles["primary_data_type"] = (
    predictor_modeling_roles["feature_name"]
    .map(feature_to_primary_type)
)

predictor_modeling_roles["missing_count"] = (
    predictor_modeling_roles["feature_name"]
    .map(features_primary_model.isna().sum())
)

predictor_modeling_roles["missing_percentage"] = (
    predictor_modeling_roles["missing_count"]
    / len(features_primary_model)
    * 100
)

predictor_modeling_roles["requires_scaling_linear_model"] = (
    predictor_modeling_roles["primary_data_type"].isin([
        "Numeric continuous",
        "Ordinal or discrete numeric",
    ])
)

predictor_modeling_roles["requires_imputation"] = (
    predictor_modeling_roles["missing_count"] > 0
)

predictor_modeling_roles["requires_one_hot_encoding"] = (
    predictor_modeling_roles["primary_data_type"].eq("Categorical")
)

predictor_modeling_roles["structurally_missing"] = (
    predictor_modeling_roles["feature_name"].isin(
        structurally_missing_features
    )
)

predictor_modeling_roles["near_constant"] = (
    predictor_modeling_roles["feature_name"].isin(
        near_constant_features
    )
)

predictor_modeling_roles["special_consideration"] = (
    predictor_modeling_roles["feature_name"]
    .map(special_consideration_map)
    .fillna("None")
)

In [8]:
# Validate predictor classifications and summarize preprocessing roles
all_classified_features = [
    feature_name
    for feature_list in primary_feature_type_groups.values()
    for feature_name in feature_list
]

duplicate_classified_features = sorted(
    pd.Series(all_classified_features)[
        pd.Series(all_classified_features).duplicated()
    ].unique()
)

unclassified_safe_predictors = sorted(
    set(leakage_safe_predictor_columns)
    - set(all_classified_features)
)

unexpected_classified_features = sorted(
    set(all_classified_features)
    - set(leakage_safe_predictor_columns)
)

numeric_type_mismatches = sorted([
    feature_name
    for feature_name in (
        numeric_continuous_features
        + numeric_binary_features
        + ordinal_discrete_numeric_features
    )
    if not pd.api.types.is_numeric_dtype(
        features_primary_model[feature_name]
    )
])

actual_missing_features = sorted(
    features_primary_model.columns[
        features_primary_model.isna().any()
    ].tolist()
)

structural_missingness_matches = (
    set(actual_missing_features)
    == set(structurally_missing_features)
)

near_constant_dominant_percentages = {
    feature_name: (
        features_primary_model[feature_name]
        .value_counts(dropna=False, normalize=True)
        .max()
        * 100
    )
    for feature_name in near_constant_features
}

near_constant_status_valid = all(
    dominant_percentage >= 99.50
    for dominant_percentage
    in near_constant_dominant_percentages.values()
)

predictor_classification_validation = pd.DataFrame({
    "validation_check": [
        "All safe predictors classified",
        "No unexpected features classified",
        "No feature assigned to multiple primary types",
        "All numeric groups contain numeric columns",
        "Structural missingness list matches observed missingness",
        "Near-constant features meet dominance threshold",
        "Final classified predictor count",
    ],
    "expected": [
        "None missing",
        "None",
        0,
        "None",
        "Exact match",
        "At least 99.50%",
        EXPECTED_PREDICTOR_COUNT,
    ],
    "actual": [
        (
            ", ".join(unclassified_safe_predictors)
            if unclassified_safe_predictors
            else "None missing"
        ),
        (
            ", ".join(unexpected_classified_features)
            if unexpected_classified_features
            else "None"
        ),
        len(duplicate_classified_features),
        (
            ", ".join(numeric_type_mismatches)
            if numeric_type_mismatches
            else "None"
        ),
        (
            "Exact match"
            if structural_missingness_matches
            else ", ".join(actual_missing_features)
        ),
        (
            "All passed"
            if near_constant_status_valid
            else "Review required"
        ),
        len(all_classified_features),
    ],
})

predictor_classification_validation["passed"] = [
    len(unclassified_safe_predictors) == 0,
    len(unexpected_classified_features) == 0,
    len(duplicate_classified_features) == 0,
    len(numeric_type_mismatches) == 0,
    structural_missingness_matches,
    near_constant_status_valid,
    len(all_classified_features) == EXPECTED_PREDICTOR_COUNT,
]

preprocessing_role_summary = pd.DataFrame({
    "primary_data_type": [
        "Numeric continuous",
        "Numeric binary",
        "Ordinal or discrete numeric",
        "Categorical",
    ],
    "feature_count": [
        len(numeric_continuous_features),
        len(numeric_binary_features),
        len(ordinal_discrete_numeric_features),
        len(categorical_features),
    ],
    "scaling_for_linear_models": [
        "Required",
        "Not required",
        "Required",
        "Not applicable",
    ],
    "imputation": [
        "Median when missing",
        "Most frequent if missing",
        "Median if missing",
        "Most frequent if missing",
    ],
    "encoding": [
        "None",
        "None",
        "None",
        "One-hot encoding",
    ],
})

special_feature_summary = predictor_modeling_roles.loc[
    predictor_modeling_roles["special_consideration"].ne("None"),
    [
        "feature_name",
        "primary_data_type",
        "missing_count",
        "missing_percentage",
        "structurally_missing",
        "near_constant",
        "special_consideration",
    ],
].copy()

display(predictor_classification_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

print("\n")

display(preprocessing_role_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["primary_data_type"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "feature_count": "{:,.0f}",
    })
)

print("\n")

display(special_feature_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["feature_name", "special_consideration"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
        {
            "selector": "th.col6",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "missing_count": "{:,.0f}",
        "missing_percentage": "{:,.2f}%",
    })
)

failed_classification_checks = (
    predictor_classification_validation.loc[
        ~predictor_classification_validation["passed"],
        "validation_check",
    ].tolist()
)

if failed_classification_checks:
    raise AssertionError(
        "Predictor classification validation failed for: "
        + ", ".join(failed_classification_checks)
    )

print(
    "\nAll predictors were classified and preprocessing roles "
    "were documented successfully.\n"
)

validation_check,expected,actual,passed
All safe predictors classified,None missing,None missing,True
No unexpected features classified,None,None,True
No feature assigned to multiple primary types,0,0,True
All numeric groups contain numeric columns,None,None,True
Structural missingness list matches observed missingness,Exact match,Exact match,True
Near-constant features meet dominance threshold,At least 99.50%,All passed,True
Final classified predictor count,53,53,True


primary_data_type,feature_count,scaling_for_linear_models,imputation,encoding
Numeric continuous,26,Required,Median when missing,None
Numeric binary,19,Not required,Most frequent if missing,None
Ordinal or discrete numeric,4,Required,Median if missing,None
Categorical,4,Not applicable,Most frequent if missing,One-hot encoding


feature_name,primary_data_type,missing_count,missing_percentage,structurally_missing,near_constant,special_consideration
feature_past_5yr_min_donation,Numeric continuous,0,0.00%,False,True,Near constant; compare retention and removal by model
feature_years_since_last_donation_past_5yr,Ordinal or discrete numeric,0,0.00%,False,False,Value 6 is a sentinel for no donation in the five-year window
feature_recent_vs_older_donation_ratio,Numeric continuous,"28,497",82.83%,True,False,Structural missingness; impute within the pipeline and consider a missingness indicator
feature_last_vs_previous_donation_ratio,Numeric continuous,0,0.00%,False,False,May contain extreme values when the previous donation is small
feature_log1p_last_vs_previous_donation_ratio,Numeric continuous,0,0.00%,False,False,Log-transformed version retained for model-specific comparison
feature_postal_code_missing_flag,Numeric binary,0,0.00%,False,True,Near constant; compare retention and removal by model
feature_address_type_other,Numeric binary,0,0.00%,False,True,Near constant; compare retention and removal by model



All predictors were classified and preprocessing roles were documented successfully.



## Primary Modeling Data and Predictor Structure

The modeling dataset was separated into three explicitly named objects:

- `tracking_donor_ids` contains the donor identifiers used only for record alignment, joins, and final prediction outputs.
- `target_primary_donor_flag` contains the binary current-fiscal-year donation target.
- `features_primary_model` contains the 53 predictors used for modeling.

The tracking identifier and target were removed from the predictor matrix before any preprocessing or model training. All three objects contain 34,403 aligned records.

The feature dictionary was used as the source of truth to construct the leakage-safe predictor list. All 53 predictors were found in the modeling export, and the list contained no duplicate names, targets, identifiers, excluded fields, or timing-sensitive features.

Four leakage-safe feature-set variants were recreated for later model comparison:

| Feature Set                     | Feature Count |
| ------------------------------- | ------------: |
| Safe Baseline Historical Set    |            10 |
| Safe Aggregate RFM Set          |             8 |
| Safe Trend-Enhanced Set         |            15 |
| Full Leakage-Safe Candidate Set |            53 |

Each feature set was validated against both the modeling dataset and feature dictionary. All expected features were present, and no unsafe or forbidden fields were included.

The 53 predictors were then separated into mutually exclusive primary data-type groups:

| Primary Data Type           | Feature Count | Planned Preprocessing                                       |
| --------------------------- | ------------: | ----------------------------------------------------------- |
| Numeric continuous          |            26 | Median imputation when needed and scaling for linear models |
| Numeric binary              |            19 | Most-frequent imputation if needed; no scaling required     |
| Ordinal or discrete numeric |             4 | Median imputation if needed and scaling for linear models   |
| Categorical                 |             4 | Most-frequent imputation and one-hot encoding               |

Tree-based pipelines will use the same feature classifications but will generally omit numeric scaling.

Several predictors require additional consideration during modeling:

- `feature_recent_vs_older_donation_ratio` contains structural missingness for 28,497 records, or 82.83% of the dataset. Its imputation must occur inside the modeling pipeline, and a missingness indicator may be evaluated.
- `feature_years_since_last_donation_past_5yr` uses the value `6` as a sentinel for no donation during the five-year historical window.
- `feature_past_5yr_min_donation`, `feature_address_type_other`, and `feature_postal_code_missing_flag` are near-constant and will be evaluated through model-specific comparisons rather than removed automatically.
- The original and log-transformed donation-ratio features were retained.
- Ratio features may contain extreme but valid values when the comparison-period donation amount is zero or very small.

All predictors were successfully classified, and the resulting preprocessing roles provide the structure needed to build separate linear and tree-based modeling pipelines.


In [9]:
# Join and validate historical donor status benchmark target
BENCHMARK_TARGET_COLUMN = "donor_indicator_flag"

EXPECTED_BENCHMARK_TARGET_COUNTS = {
    0: 13_034,
    1: 21_369,
}

benchmark_target_source = pd.read_csv(
    CLEANED_DONOR_DATA_PATH,
    usecols=[
        TRACKING_IDENTIFIER_COLUMN,
        BENCHMARK_TARGET_COLUMN,
    ],
)

benchmark_target_join = modeling_data[
    [TRACKING_IDENTIFIER_COLUMN]
].merge(
    benchmark_target_source,
    on=TRACKING_IDENTIFIER_COLUMN,
    how="left",
    validate="one_to_one",
    indicator=True,
)

target_benchmark_donor_indicator_flag = benchmark_target_join[
    BENCHMARK_TARGET_COLUMN
].copy()

benchmark_target_distribution = (
    target_benchmark_donor_indicator_flag
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis("target_class")
    .reset_index(name="record_count")
)

benchmark_target_distribution["percentage"] = (
    benchmark_target_distribution["record_count"]
    / len(target_benchmark_donor_indicator_flag)
    * 100
)

benchmark_target_distribution["expected_record_count"] = (
    benchmark_target_distribution["target_class"]
    .map(EXPECTED_BENCHMARK_TARGET_COUNTS)
)

benchmark_target_distribution["matches_expected"] = (
    benchmark_target_distribution["record_count"]
    == benchmark_target_distribution["expected_record_count"]
)

actual_benchmark_target_counts = (
    benchmark_target_distribution
    .set_index("target_class")["record_count"]
    .to_dict()
)

benchmark_in_primary_feature_sets = sorted({
    feature_set_name
    for feature_set_name, feature_list in feature_set_variants.items()
    if BENCHMARK_TARGET_COLUMN in feature_list
})

benchmark_target_validation = pd.DataFrame({
    "validation_check": [
        "Benchmark source identifier missing count",
        "Benchmark source identifier duplicate count",
        "Join matched record count",
        "Joined row count",
        "Benchmark target missing count",
        "Benchmark target contains only 0 and 1",
        "Benchmark target class 0 count",
        "Benchmark target class 1 count",
        "Benchmark target excluded from primary predictor matrix",
        "Benchmark target excluded from safe predictor list",
        "Benchmark target excluded from primary feature sets",
    ],
    "expected": [
        0,
        0,
        EXPECTED_ROW_COUNT,
        EXPECTED_ROW_COUNT,
        0,
        "True",
        EXPECTED_BENCHMARK_TARGET_COUNTS[0],
        EXPECTED_BENCHMARK_TARGET_COUNTS[1],
        "True",
        "True",
        "True",
    ],
    "actual": [
        benchmark_target_source[
            TRACKING_IDENTIFIER_COLUMN
        ].isna().sum(),
        benchmark_target_source[
            TRACKING_IDENTIFIER_COLUMN
        ].duplicated().sum(),
        benchmark_target_join["_merge"].eq("both").sum(),
        len(benchmark_target_join),
        target_benchmark_donor_indicator_flag.isna().sum(),
        str(
            set(
                target_benchmark_donor_indicator_flag
                .dropna()
                .unique()
            ).issubset({0, 1})
        ),
        actual_benchmark_target_counts[0],
        actual_benchmark_target_counts[1],
        str(
            BENCHMARK_TARGET_COLUMN
            not in features_primary_model.columns
        ),
        str(
            BENCHMARK_TARGET_COLUMN
            not in leakage_safe_predictor_columns
        ),
        str(len(benchmark_in_primary_feature_sets) == 0),
    ],
})

benchmark_target_validation["passed"] = [
    benchmark_target_source[
        TRACKING_IDENTIFIER_COLUMN
    ].isna().sum() == 0,
    benchmark_target_source[
        TRACKING_IDENTIFIER_COLUMN
    ].duplicated().sum() == 0,
    benchmark_target_join["_merge"].eq("both").sum()
    == EXPECTED_ROW_COUNT,
    len(benchmark_target_join) == EXPECTED_ROW_COUNT,
    target_benchmark_donor_indicator_flag.isna().sum() == 0,
    set(
        target_benchmark_donor_indicator_flag
        .dropna()
        .unique()
    ).issubset({0, 1}),
    actual_benchmark_target_counts[0]
    == EXPECTED_BENCHMARK_TARGET_COUNTS[0],
    actual_benchmark_target_counts[1]
    == EXPECTED_BENCHMARK_TARGET_COUNTS[1],
    BENCHMARK_TARGET_COLUMN
    not in features_primary_model.columns,
    BENCHMARK_TARGET_COLUMN
    not in leakage_safe_predictor_columns,
    len(benchmark_in_primary_feature_sets) == 0,
]

display(benchmark_target_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

print("")

display(benchmark_target_distribution.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_table_styles([{
        "selector": "th",
        "props": [
            ("text-align", "center"),
            ("padding", "8px 16px"),
        ],
    }])
    .format({
        "target_class": "{:,.0f}",
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
        "expected_record_count": "{:,.0f}",
    })
)

failed_benchmark_target_checks = benchmark_target_validation.loc[
    ~benchmark_target_validation["passed"],
    "validation_check",
].tolist()

if failed_benchmark_target_checks:
    raise AssertionError(
        "Benchmark target validation failed for: "
        + ", ".join(failed_benchmark_target_checks)
    )

print(
    "\nHistorical donor-status benchmark target joined "
    "and validated successfully."
)

validation_check,expected,actual,passed
Benchmark source identifier missing count,0,0,True
Benchmark source identifier duplicate count,0,0,True
Join matched record count,"34,403","34,403",True
Joined row count,"34,403","34,403",True
Benchmark target missing count,0,0,True
Benchmark target contains only 0 and 1,True,True,True
Benchmark target class 0 count,"13,034","13,034",True
Benchmark target class 1 count,"21,369","21,369",True
Benchmark target excluded from primary predictor matrix,True,True,True
Benchmark target excluded from safe predictor list,True,True,True


target_class,record_count,percentage,expected_record_count,matches_expected
0,"13,034",37.89%,"13,034",True
1,"21,369",62.11%,"21,369",True



Historical donor-status benchmark target joined and validated successfully.


In [10]:
# Audit benchmark target against donation based construction rules
historical_donation_columns = [
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
]

benchmark_audit_source = pd.read_csv(
    CLEANED_DONOR_DATA_PATH,
    usecols=[
        TRACKING_IDENTIFIER_COLUMN,
        BENCHMARK_TARGET_COLUMN,
        "cumulative_donation_amount",
        *historical_donation_columns,
    ],
)

benchmark_construction_audit = tracking_donor_ids.to_frame(
    name=TRACKING_IDENTIFIER_COLUMN
).merge(
    benchmark_audit_source,
    on=TRACKING_IDENTIFIER_COLUMN,
    how="left",
    validate="one_to_one",
)

benchmark_construction_audit["historical_positive_year_count"] = (
    benchmark_construction_audit[historical_donation_columns]
    .gt(0)
    .sum(axis=1)
)

benchmark_construction_audit["historical_5yr_total"] = (
    benchmark_construction_audit[historical_donation_columns]
    .sum(axis=1)
)

benchmark_actual_target = benchmark_construction_audit[
    BENCHMARK_TARGET_COLUMN
].astype("int8")

historical_missing_rows = (
    benchmark_construction_audit[historical_donation_columns]
    .isna()
    .any(axis=1)
    .sum()
)

cumulative_missing_rows = (
    benchmark_construction_audit[
        "cumulative_donation_amount"
    ]
    .isna()
    .sum()
)

benchmark_candidate_rules = {
    "Any positive historical donation": (
        benchmark_construction_audit[
            "historical_positive_year_count"
        ].ge(1).astype("int8"),
        historical_missing_rows,
    ),
    "Positive five-year historical total": (
        benchmark_construction_audit[
            "historical_5yr_total"
        ].gt(0).astype("int8"),
        historical_missing_rows,
    ),
    "Donated in at least 2 historical years": (
        benchmark_construction_audit[
            "historical_positive_year_count"
        ].ge(2).astype("int8"),
        historical_missing_rows,
    ),
    "Donated in at least 3 historical years": (
        benchmark_construction_audit[
            "historical_positive_year_count"
        ].ge(3).astype("int8"),
        historical_missing_rows,
    ),
    "Donated in at least 4 historical years": (
        benchmark_construction_audit[
            "historical_positive_year_count"
        ].ge(4).astype("int8"),
        historical_missing_rows,
    ),
    "Donated in all 5 historical years": (
        benchmark_construction_audit[
            "historical_positive_year_count"
        ].eq(5).astype("int8"),
        historical_missing_rows,
    ),
    "Positive cumulative donation": (
        benchmark_construction_audit[
            "cumulative_donation_amount"
        ].gt(0).astype("int8"),
        cumulative_missing_rows,
    ),
}

benchmark_reconstruction_records = []

for rule_name, (
    candidate_target,
    source_missing_rows,
) in benchmark_candidate_rules.items():

    disagreement_count = int(
        candidate_target.ne(benchmark_actual_target).sum()
    )

    benchmark_reconstruction_records.append({
        "candidate_rule": rule_name,
        "source_missing_rows": source_missing_rows,
        "predicted_positive_count": candidate_target.sum(),
        "predicted_positive_percentage": (
            candidate_target.mean() * 100
        ),
        "disagreement_count": disagreement_count,
        "agreement_percentage": (
            candidate_target.eq(benchmark_actual_target).mean()
            * 100
        ),
        "exact_reconstruction": (
            source_missing_rows == 0
            and disagreement_count == 0
        ),
    })

benchmark_reconstruction_results = pd.DataFrame(
    benchmark_reconstruction_records
)

exact_benchmark_reconstruction_rules = (
    benchmark_reconstruction_results.loc[
        benchmark_reconstruction_results[
            "exact_reconstruction"
        ],
        "candidate_rule",
    ].tolist()
)

display(benchmark_reconstruction_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["candidate_rule"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "source_missing_rows": "{:,.0f}",
        "predicted_positive_count": "{:,.0f}",
        "predicted_positive_percentage": "{:,.2f}%",
        "disagreement_count": "{:,.0f}",
        "agreement_percentage": "{:,.2f}%",
    })
)

if exact_benchmark_reconstruction_rules:
    print(
        "\nExact benchmark reconstruction rule(s): "
        + ", ".join(exact_benchmark_reconstruction_rules)
    )
else:
    print(
        "No tested donation-based rule exactly reconstructs "
        "donor_indicator_flag."
    )

candidate_rule,source_missing_rows,predicted_positive_count,predicted_positive_percentage,disagreement_count,agreement_percentage,exact_reconstruction
Any positive historical donation,0,"9,087",26.41%,"12,282",64.30%,False
Positive five-year historical total,0,"9,087",26.41%,"12,282",64.30%,False
Donated in at least 2 historical years,0,"1,894",5.51%,"19,475",43.39%,False
Donated in at least 3 historical years,0,407,1.18%,"20,962",39.07%,False
Donated in at least 4 historical years,0,60,0.17%,"21,309",38.06%,False
Donated in all 5 historical years,0,10,0.03%,"21,359",37.92%,False
Positive cumulative donation,0,"21,369",62.11%,0,100.00%,True



Exact benchmark reconstruction rule(s): Positive cumulative donation


In [11]:
# Define and validate conservative benchmark predictor set
conservative_benchmark_predictor_columns = [
    "donor_age",
    "is_alumnus_flag",
    "is_parent_flag",
    "feature_age_group",
    "feature_age_decade",
    "feature_age_squared",
    "feature_gender_identity",
    "feature_gender_missing_flag",
    "feature_gender_unknown_flag",
    "feature_preferred_address_type",
    "feature_preferred_address_missing_flag",
    "feature_is_business_address",
    "feature_is_home_address",
    "feature_is_campus_address",
    "feature_address_type_other",
    "feature_postal_code_missing_flag",
]

if exact_benchmark_reconstruction_rules:
    benchmark_predictor_strategy = (
        "Donation-based construction identified; historical donation "
        "features and derivatives excluded to avoid circular prediction"
    )
else:
    benchmark_predictor_strategy = (
        "Construction not verified; conservative non-donation "
        "predictor set used"
    )

benchmark_predictor_columns = (
    conservative_benchmark_predictor_columns.copy()
)

features_benchmark_model = features_primary_model[
    benchmark_predictor_columns
].copy()

benchmark_excluded_donation_predictors = sorted(
    set(leakage_safe_predictor_columns)
    - set(benchmark_predictor_columns)
)

missing_benchmark_predictors = sorted(
    set(benchmark_predictor_columns)
    - set(features_primary_model.columns)
)

non_safe_benchmark_predictors = sorted(
    set(benchmark_predictor_columns)
    - set(leakage_safe_predictor_columns)
)

duplicate_benchmark_predictors = sorted(
    pd.Series(benchmark_predictor_columns)[
        pd.Series(
            benchmark_predictor_columns
        ).duplicated()
    ].unique()
)

benchmark_forbidden_predictors = sorted(
    set(benchmark_predictor_columns).intersection({
        TRACKING_IDENTIFIER_COLUMN,
        PRIMARY_TARGET_COLUMN,
        BENCHMARK_TARGET_COLUMN,
        "cumulative_donation_amount",
        *historical_donation_columns,
    })
)

benchmark_predictor_validation = pd.DataFrame({
    "validation_check": [
        "Benchmark predictors exist in primary feature matrix",
        "Benchmark predictors are leakage-safe",
        "No duplicate benchmark predictors",
        "No identifier or target included",
        "Historical donation fields excluded",
        "Benchmark predictor count",
    ],
    "expected": [
        "None missing",
        "None",
        0,
        "None",
        "Excluded",
        len(conservative_benchmark_predictor_columns),
    ],
    "actual": [
        (
            ", ".join(missing_benchmark_predictors)
            if missing_benchmark_predictors
            else "None missing"
        ),
        (
            ", ".join(non_safe_benchmark_predictors)
            if non_safe_benchmark_predictors
            else "None"
        ),
        len(duplicate_benchmark_predictors),
        (
            ", ".join(benchmark_forbidden_predictors)
            if benchmark_forbidden_predictors
            else "None"
        ),
        (
            "Excluded"
            if not benchmark_forbidden_predictors
            else "Review required"
        ),
        len(benchmark_predictor_columns),
    ],
})

benchmark_predictor_validation["passed"] = [
    len(missing_benchmark_predictors) == 0,
    len(non_safe_benchmark_predictors) == 0,
    len(duplicate_benchmark_predictors) == 0,
    len(benchmark_forbidden_predictors) == 0,
    len(benchmark_forbidden_predictors) == 0,
    (
        len(benchmark_predictor_columns)
        == len(conservative_benchmark_predictor_columns)
    ),
]

benchmark_predictor_summary = pd.DataFrame({
    "predictor_strategy": [benchmark_predictor_strategy],
    "included_predictor_count": [
        len(benchmark_predictor_columns)
    ],
    "excluded_donation_predictor_count": [
        len(benchmark_excluded_donation_predictors)
    ],
})

display(benchmark_predictor_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

print("")

display(benchmark_predictor_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["predictor_strategy"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "included_predictor_count": "{:,.0f}",
        "excluded_donation_predictor_count": "{:,.0f}",
    })
)

failed_benchmark_predictor_checks = (
    benchmark_predictor_validation.loc[
        ~benchmark_predictor_validation["passed"],
        "validation_check",
    ].tolist()
)

if failed_benchmark_predictor_checks:
    raise AssertionError(
        "Benchmark predictor validation failed for: "
        + ", ".join(failed_benchmark_predictor_checks)
    )

print(
    "\nHistorical donor-status benchmark predictor strategy "
    "validated successfully."
)

validation_check,expected,actual,passed
Benchmark predictors exist in primary feature matrix,None missing,None missing,True
Benchmark predictors are leakage-safe,None,None,True
No duplicate benchmark predictors,0,0,True
No identifier or target included,None,None,True
Historical donation fields excluded,Excluded,Excluded,True
Benchmark predictor count,16,16,True


predictor_strategy,included_predictor_count,excluded_donation_predictor_count
Donation-based construction identified; historical donation features and derivatives excluded to avoid circular prediction,16,37



Historical donor-status benchmark predictor strategy validated successfully.


## Historical Donor-Status Benchmark Preparation

The original `donor_indicator_flag` was added as a separate benchmark target by joining it from the cleaned donor dataset using `donor_unique_id`.

The benchmark target validation confirmed:

- The join was one-to-one.
- The row count remained 34,403.
- No benchmark target values were missing.
- The target contained only binary values.
- The class distribution matched the Phase 4 documentation.
- The benchmark target was not added to `features_primary_model`, the leakage-safe predictor list, or any primary-model feature set.

The benchmark target distribution is:

| Target Class | Records | Percentage |
| ------------ | ------: | ---------: |
| 0            |  13,034 |     37.89% |
| 1            |  21,369 |     62.11% |

Because the construction of `donor_indicator_flag` was previously uncertain, several plausible donation-based rules were tested.

The five-year historical donation rules did not reproduce the target exactly. For example, classifying anyone with at least one positive donation during the five completed historical fiscal years as a donor produced only 64.30% agreement.

However, the following rule reproduced the benchmark target exactly:

`cumulative_donation_amount > 0`

This rule produced:

- 21,369 positive records
- 0 disagreements
- 100.00% agreement with `donor_indicator_flag`

This confirms that, within the available dataset, `donor_indicator_flag` functions as an indicator of whether cumulative donation is positive. It therefore represents historical donor status rather than the forward-looking current-fiscal-year outcome used by the primary model.

To avoid circular prediction, the benchmark model excludes:

- `cumulative_donation_amount`
- The five individual historical donation amount fields
- Donation aggregates
- Donation frequency and streak features
- Donation recency features
- Donation trend and trajectory features
- Log-transformed donation features

These fields were used only to investigate how the benchmark target was constructed and are excluded from the benchmark predictor set to avoid circular prediction.

A conservative benchmark predictor set containing 16 non-donation features was therefore defined. These consist of demographic, alumni and parent status, age representations, gender features, preferred-address features, and postal-code missingness.

The benchmark workflow will remain separate from the primary future-donation model because the two targets represent different prediction problems, use different predictor restrictions, and have different business interpretations.


In [12]:
# Define modeling experiment configuration
TEST_SIZE = 0.20
CV_FOLDS = 5
PRIMARY_SCORING = "average_precision"
DEFAULT_PROBABILITY_THRESHOLD = 0.50
OUTREACH_CAPACITY_PERCENTAGES = (0.01, 0.05, 0.10, 0.20)

modeling_experiment_config = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "cv_folds": CV_FOLDS,
    "n_jobs": N_JOBS,
    "primary_scoring": PRIMARY_SCORING,
    "default_probability_threshold": DEFAULT_PROBABILITY_THRESHOLD,
    "outreach_capacity_percentages": OUTREACH_CAPACITY_PERCENTAGES,
}

modeling_experiment_summary = pd.DataFrame({
    "setting": [
        "Random state",
        "Test-set proportion",
        "Cross-validation folds",
        "Parallel-processing jobs",
        "Primary scoring metric",
        "Default probability threshold",
        "Outreach-capacity percentages",
    ],
    "value": [
        f"{RANDOM_STATE:,}",
        f"{TEST_SIZE:.2%}",
        f"{CV_FOLDS:,}",
        str(N_JOBS),
        "PR-AUC (average_precision)",
        f"{DEFAULT_PROBABILITY_THRESHOLD:.2f}",
        ", ".join(
            f"{capacity:.0%}"
            for capacity in OUTREACH_CAPACITY_PERCENTAGES
        ),
    ],
})

display(modeling_experiment_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["setting"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
)

assert 0 < TEST_SIZE < 1
assert CV_FOLDS >= 2
assert 0 <= DEFAULT_PROBABILITY_THRESHOLD <= 1
assert all(
    0 < capacity <= 1
    for capacity in OUTREACH_CAPACITY_PERCENTAGES
)

print(
    "\nModeling experiment configuration defined successfully."
)

setting,value
Random state,121
Test-set proportion,20.00%
Cross-validation folds,5
Parallel-processing jobs,-1
Primary scoring metric,PR-AUC (average_precision)
Default probability threshold,0.50
Outreach-capacity percentages,"1%, 5%, 10%, 20%"



Modeling experiment configuration defined successfully.
